In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Phase1") \
    .getOrCreate()

In [0]:
from pyspark.sql.functions import *

In [0]:
customers = spark.read.option("header","true").option("inferSchema","true").csv("/Workspace/Users/manthenapoojith@gmail.com/customers.csv")

orders = spark.read.option("header","true").option("inferSchema","true").csv("/Workspace/Users/manthenapoojith@gmail.com/orders.csv")

display(customers)
display(orders)

customer_id,name,city
C101,Aadhvik Rao,Bengaluru
C102,Ishani Mehta,Mumbai
C103,Vihaan Kapoor,Delhi
C104,Myra Nair,Hyderabad
C105,Rudra Sen,Kolkata
C106,Kiara Thomas,Chennai
C107,Reyansh Gupta,Pune
C108,Anika Reddy,Hyderabad
C109,Vivaan Joseph,Kochi
C110,Tara Malhotra,Jaipur


order_id,customer_id,order_amount
O001,C101,2500
O002,C102,4000
O003,C101,3500
O004,C103,2000
O005,C104,6000
O006,C104,3000
O007,C106,4500
O008,C107,7000
O009,C107,1000
O010,C108,8000


In [0]:
customers.printSchema()

orders.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_amount: integer (nullable = true)



In [0]:
customers = customers.dropna(subset=["customer_id"])

orders = orders.dropna(subset=["customer_id"])

In [0]:
customers.createOrReplaceTempView("customers")

orders.createOrReplaceTempView("orders")

Execrise 1

In [0]:
%sql
SELECT
c.customer_id,
c.name,
SUM(o.order_amount) AS total_amount
FROM customers c
JOIN orders o
ON c.customer_id=o.customer_id
GROUP BY c.customer_id,c.name;

customer_id,name,total_amount
C101,Aadhvik Rao,6000
C108,Anika Reddy,12000
C110,Tara Malhotra,5000
C107,Reyansh Gupta,8000
C102,Ishani Mehta,4000
C106,Kiara Thomas,4500
C103,Vihaan Kapoor,2000
C104,Myra Nair,9000


In [0]:
display(
customers.join(
orders,
"customer_id"
).groupBy(
"customer_id",
"name"
).agg(
sum("order_amount").alias("total_amount")
)
)

customer_id,name,total_amount
C101,Aadhvik Rao,6000
C108,Anika Reddy,12000
C110,Tara Malhotra,5000
C107,Reyansh Gupta,8000
C102,Ishani Mehta,4000
C106,Kiara Thomas,4500
C103,Vihaan Kapoor,2000
C104,Myra Nair,9000


2. Top 3 customers by total spend

In [0]:
%sql
SELECT
c.customer_id,
c.name,
SUM(order_amount) total_spend
FROM customers c
JOIN orders o
ON c.customer_id=o.customer_id
GROUP BY c.customer_id,c.name
ORDER BY total_spend DESC
LIMIT 3;

customer_id,name,total_spend
C108,Anika Reddy,12000
C104,Myra Nair,9000
C107,Reyansh Gupta,8000


In [0]:
display(
customers.join(
orders,
"customer_id"
).groupBy(
"customer_id",
"name"
).agg(
sum("order_amount").alias("total_spend")
).orderBy(
desc("total_spend")
).limit(3)
)

customer_id,name,total_spend
C108,Anika Reddy,12000
C104,Myra Nair,9000
C107,Reyansh Gupta,8000


3. Customers with no orders

In [0]:
%sql
SELECT *
FROM customers c
LEFT JOIN orders o
ON c.customer_id=o.customer_id
WHERE o.customer_id IS NULL;

customer_id,name,city,order_id,customer_id,order_amount
C105,Rudra Sen,Kolkata,null,null,null
C109,Vivaan Joseph,Kochi,null,null,null


In [0]:
display(
customers.join(
orders,
"customer_id",
"left"
).filter(
orders.customer_id.isNull()
)
)

customer_id,name,city,order_id,order_amount
C105,Rudra Sen,Kolkata,null,null
C109,Vivaan Joseph,Kochi,null,null


4. City-wise total revenue


In [0]:
%sql
SELECT
city,
SUM(order_amount) revenue
FROM customers c
JOIN orders o
ON c.customer_id=o.customer_id
GROUP BY city;

city,revenue
Delhi,2000
Chennai,4500
Jaipur,5000
Hyderabad,21000
Pune,8000
Mumbai,4000
Bengaluru,6000


In [0]:
display(
customers.join(
orders,
"customer_id"
).groupBy(
"city"
).agg(
sum("order_amount").alias("revenue")
)
)

city,revenue
Delhi,2000
Chennai,4500
Jaipur,5000
Hyderabad,21000
Pune,8000
Mumbai,4000
Bengaluru,6000


5. Average order amount per customer

In [0]:
%sql
SELECT
customer_id,
AVG(order_amount)
FROM orders
GROUP BY customer_id;

customer_id,AVG(order_amount)
C103,2000.0
C107,4000.0
C108,4000.0
C110,5000.0
C104,4500.0
C106,4500.0
C101,3000.0
C102,4000.0


In [0]:
display(
orders.groupBy(
"customer_id"
).agg(
avg("order_amount").alias("average_order")
)
)

customer_id,average_order
C103,2000.0
C107,4000.0
C108,4000.0
C110,5000.0
C104,4500.0
C106,4500.0
C101,3000.0
C102,4000.0


6. Customers with more than one order

In [0]:
%sql
SELECT
customer_id,
COUNT(*)
FROM orders
GROUP BY customer_id
HAVING COUNT(*)>1;

customer_id,COUNT(*)
C107,2
C108,3
C104,2
C101,2


In [0]:
display(
orders.groupBy(
"customer_id"
).agg(
count("*").alias("orders")
).filter(
col("orders")>1
)
)

customer_id,orders
C107,2
C108,3
C104,2
C101,2


7. Sort customers by total spend descending

In [0]:
%sql
SELECT
c.customer_id,
c.name,
SUM(order_amount) total_spend
FROM customers c
JOIN orders o
ON c.customer_id=o.customer_id
GROUP BY c.customer_id,c.name
ORDER BY total_spend DESC;

customer_id,name,total_spend
C108,Anika Reddy,12000
C104,Myra Nair,9000
C107,Reyansh Gupta,8000
C101,Aadhvik Rao,6000
C110,Tara Malhotra,5000
C106,Kiara Thomas,4500
C102,Ishani Mehta,4000
C103,Vihaan Kapoor,2000


In [0]:
display(
customers.join(
orders,
"customer_id"
).groupBy(
"customer_id",
"name"
).agg(
sum("order_amount").alias("total_spend")
).orderBy(
desc("total_spend")
)
)

customer_id,name,total_spend
C108,Anika Reddy,12000
C104,Myra Nair,9000
C107,Reyansh Gupta,8000
C101,Aadhvik Rao,6000
C110,Tara Malhotra,5000
C106,Kiara Thomas,4500
C102,Ishani Mehta,4000
C103,Vihaan Kapoor,2000
